# Homework 4 — Experiments

**Deadline:** Tuesday, June 2, 2026, 23:59 (Barcelona time) &middot; **Total points:** 10 (+1 bonus) &middot; **Solo work**

Hands-on with an A/B experiment end-to-end, framed as a product case. Six parts and a bonus.

**Grading (10 + 1)**

| Part | Topic | Points |
|---|---|---|
| 1 | Planning (MDE + weeks per metric) | 3 |
| 2 | Health (SRM) | 1 |
| 3 | Analysis (readout table + Bonferroni + 2–3 sentence interpretation) | 3 |
| 4 | CUPED (variance reduction table) | 1 |
| ★ 5 | Simulation: peeking | 1 |
| ★ 6 | Simulation: power curve | 1 |
| Bonus | Delta-method CI for the relative lift | +1 |

Parts 1–4 are graded on completion and the shape of your reasoning. The `check_answer` calls in those parts are self-checks — their PASS / FAIL does not enter the grade. Parts 5–6 and the bonus are graded on `check_answer` PASS / FAIL.

Hand in this notebook with your code filled in and the requested plots rendered.

**Libraries.** Use whatever you prefer for visualisation: `matplotlib`, `seaborn`, `plotly`. The grader checks numerics only. Plots are for your understanding and for me to read.

**On AI.** Use it for syntax, docs, and debugging. The math, the structure, and the interpretation should be yours.

---

## The product case

You are a product analyst at a subscription SaaS product. The product currently has a single paid plan, **Pro**, starting at $15 / month. Free users get a limited feature set; everyone who pays is on Pro.

The Growth PM, Olga, proposes a **new Lite tier at $5 / month** alongside Pro. Lite unlocks core paid features (the things free users hit the paywall on most often) but excludes the advanced features that distinguish Pro. Her thesis:

> The current paywall is too high. Many free users would pay something, but not $15. With Lite at $5 we widen the paying funnel, more free users convert, and the increase in paying base outweighs the lower per-user revenue. Net: average revenue per user (ARPU) grows by at least **5% relative**.

The risk is exactly the symmetric one: existing Pro users could downgrade to Lite, and the new Lite payers contribute less than Pro payers do, so per-payer revenue (ARPPU) drops. Whether the net ARPU lift is positive depends on the relative magnitudes.

Engineering can run a 50 / 50 split on new sign-ups. Olga originally budgeted **two weeks** but is open to extending if the planning math calls for it. She wants one structured analyst report from you.

Your steps, in order:

1. **Planning.** Compute the MDE and the required weeks for every metric we plan to read out. Is the original 2-week budget enough for the 5% target on ARPU? If not, how long do we need?
2. **Health.** Once data lands, check for a sample ratio mismatch.
3. **Analysis and interpretation.** Build one results table covering revenue, the proxies, the guardrails, and ARPPU. Apply Bonferroni, decide ship or kill, and write a short interpretation of what happened with the Lite-tier launch.
4. **Variance reduction.** Build a small CUPED table: try `usage_pre` as a covariate for `usage_minutes` and for `revenue`. Decide where it is worth applying.
5. **★ Simulation: peeking** — show that checking the p-value daily inflates the FPR.
6. **★ Simulation: power curve** — show how power grows with $n$ at a fixed effect.
7. **Bonus.** Re-express the revenue lift as a percentage with a proper CI via the delta method.

---

## A note on terminology

Throughout this assignment we use the **asymptotic Z-test** for comparing two means: under $H_0$, the standardised statistic $Z = (\bar X_T - \bar X_C) / \widehat{\text{SE}}$ is approximately $\mathcal{N}(0, 1)$ by CLT. We do not assume the data is Normal. At our sample sizes, `scipy.stats.ttest_ind(..., equal_var=False)` is numerically identical to the asymptotic Z-test, so we use it as a stand-in. We will refer to it as the **asymptotic Z-test** throughout and name variables `z_stat`, `z_p`.

---

## Setup

Run the cell below once. It loads the libraries and the hidden grader.


In [ ]:
# DO NOT MODIFY
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st

import _grader
importlib.reload(_grader)  # force fresh grader if it was updated after kernel start
from _grader import check_answer

print("ready")


---

## Load the data

`ab_data.csv` is one row per user. Columns:

| Column | Type | Meaning |
|---|---|---|
| `user_id` | int | unique user identifier |
| `group` | str | `"control"` or `"treatment"` |
| `usage_pre` | float | usage minutes in the week **before** launch (CUPED covariate) |
| `usage_minutes` | float | usage minutes during the experiment — proxy metric |
| `is_paying` | int (0/1) | did the user hold a paid subscription during the window — proxy metric |
| `revenue` | float | revenue from the user during the window (0 for non-payers) — **success metric** |
| `latency_ms` | float | average request latency in ms — guardrail |
| `error_rate_flag` | int (0/1) | did the user hit an error — guardrail |
| `support_tickets` | int | number of support tickets the user filed — guardrail |

A derived metric you will use later: **ARPPU** = mean revenue conditional on `is_paying == 1`.

> **Reading the story.** The CSV contains the **full 8-week run** (≈ 80k users) because the team agreed to extend the experiment after the planning step. For Part 1 you will compute the MDE as if you were still on the original 2-week budget (n = 10,000 per group) — that is the planning that justified the extension. From Part 2 onward you work with the actual 8-week data that landed.


In [ ]:
df = pd.read_csv("ab_data.csv")
ctrl = df[df["group"] == "control"]
treat = df[df["group"] == "treatment"]
print(f"n_control = {len(ctrl)}, n_treatment = {len(treat)}")
df.head()


---

## Part 1 — Planning: MDE and weeks to run

The experiment needs to be powered for **every** metric we report on, not just the success metric. Each metric has its own $\alpha$, $\beta$, and the smallest effect the business cares about.

Two formulas you will use:

$$\text{MDE}(n) = \left( z_{1 - \alpha/2} + z_{1 - \beta} \right) \cdot \sigma \cdot \sqrt{\frac{2}{n}}, \qquad n^* = \frac{2\,\sigma^2 \left( z_{1 - \alpha/2} + z_{1 - \beta} \right)^2}{\Delta^2}$$

**Where does the factor of $2$ come from?** It is the standard error of the **difference** of two independent group means under equal $n$ and equal $\sigma$:

$$\widehat{\text{Var}}(\bar X_T - \bar X_C) = \frac{\sigma^2}{n_T} + \frac{\sigma^2}{n_C} = \frac{2 \sigma^2}{n} \;\Longrightarrow\; \widehat{\text{SE}} = \sigma \sqrt{\tfrac{2}{n}}$$

Each group contributes $\sigma^2 / n$ to the variance of the difference, so the SE of the gap is $\sqrt{2}$ times the SE of a single group mean. MDE is the smallest $\Delta$ such that $\Delta / \widehat{\text{SE}}$ clears both $z_{1-\alpha/2}$ (rejection threshold) and $z_{1-\beta}$ (detection threshold) — hence the $2$ under the square root.

Express MDE in **both** absolute units and as a **percent** of the control mean. The percent form makes numbers comparable across metrics that live on very different scales.

The platform delivers roughly $5{,}000$ new users per group per week, so the planning answer should come back in **weeks** rather than raw $n$:

$$\text{weeks needed} = \left\lceil \frac{n^*}{5000} \right\rceil$$

The PM has budgeted **2 weeks**. Use $\alpha = 0.05$ and $\beta = 0.20$ for every metric (so $k = z_{0.975} + z_{0.80} \approx 2.80$ across the board). The plan:

| Metric | role | kind | target effect (absolute) |
|---|---|---|---|
| `arpu` | success — mean revenue across all users | all | 0.75 |
| `arppu` | proxy — mean revenue conditional on `is_paying = 1` | payers | 3.0 |
| `usage_minutes` | proxy | all | 2.0 |
| `is_paying` | proxy | all | 0.015 |
| `latency_ms` | guardrail | all | 5.0 |
| `error_rate_flag` | guardrail | all | 0.01 |

> **ARPPU is conditional.** It is only defined on paying users, so it accumulates more slowly. With a control paying rate of $\approx 14\%$ that is roughly $5000 \times 0.14 \approx 700$ payers per group per week. For ARPPU use `users_per_week_arppu = paying_rate_ctrl * USERS_PER_WEEK` instead of the full $5000$, and compute $\sigma$ on the payer-only subset. Same MDE formula, different denominator under the square root.

> **What to expect.** Secondary metrics (`usage_minutes`, `is_paying`) and guardrails are closer to the user action and have lower variance, so they reach their MDE in a small number of weeks. **`revenue` is the noisy summary at the end of the chain.** It needs many more users to detect the same relative effect. The ship-or-kill decision is anchored on `revenue` (the PM's hypothesis is about ARPU), so the binding constraint is the success metric's weeks. Do not get fooled by proxies that resolve fast.

For each row, take $\sigma$ from the **control** group. Compute:

1. `target_pct` — target effect as a percent of the control mean
2. `mde_abs_planned`, `mde_pct_planned` — MDE evaluated at the PM's planned $n = 2 \times 5000 = 10{,}000$ per group
3. `n_star` — required $n$ per group to reach the target effect
4. `weeks` — $\lceil n^* / 5000 \rceil$

The experiment is properly sized only if `PLANNED_WEEKS` $\ge \max(\text{weeks})$ across metrics.

**Tasks:**
- `1_mde_abs_arpu`, `1_mde_pct_arpu` — MDE on ARPU at the planned $n$, both forms
- `1_weeks_arpu`, `1_weeks_arppu`, `1_weeks_usage`, `1_weeks_paying`, `1_weeks_latency`, `1_weeks_error` — weeks per metric
- `1_max_weeks` — the binding constraint
- `1_can_run` — `1` if the 2-week plan is enough, else `0`


In [ ]:
USERS_PER_WEEK = 5000
PLANNED_WEEKS = 2
ALPHA, BETA = 0.05, 0.20

PLAN = pd.DataFrame({
    "metric":        ["arpu", "arppu",  "usage_minutes", "is_paying", "latency_ms", "error_rate_flag"],
    "kind":          ["all",  "payers", "all",           "all",       "all",        "all"],
    "target_effect": [0.75,   3.0,      2.0,             0.015,       5.0,          0.01],
})

# Build a DETAILS DataFrame: one row per metric with MDE at the planned n
# (abs + pct), n_star and weeks. For kind == "payers", σ comes from the
# payer-only subset and users_per_week_arppu = paying_rate_ctrl * USERS_PER_WEEK.

weeks_dict = ...
mde_abs_arpu = ...
mde_pct_arpu = ...

max_weeks = ...
can_run = ...

check_answer("1_mde_abs_arpu", mde_abs_arpu)
check_answer("1_mde_pct_arpu", mde_pct_arpu)
check_answer("1_weeks_arpu", weeks_dict["arpu"])
check_answer("1_weeks_arppu", weeks_dict["arppu"])
check_answer("1_weeks_usage", weeks_dict["usage_minutes"])
check_answer("1_weeks_paying", weeks_dict["is_paying"])
check_answer("1_weeks_latency", weeks_dict["latency_ms"])
check_answer("1_weeks_error", weeks_dict["error_rate_flag"])
check_answer("1_max_weeks", max_weeks)
check_answer("1_can_run", can_run)


---

## After Part 1 — what happened next

You walk the planning table to Olga and the engineering lead. ARPU takes about eight weeks to clear its MDE at the 5% target, and the proxies and guardrails resolve faster. Two weeks would have been short of what ARPU needs, so the team agrees to **extend the run to eight weeks**. That gives ARPU enough time to mature. The proxies and guardrails are read as they cross their own MDEs. **Launch approved.**

Eight weeks later, the data is on your dashboard. You move on to the health check.


---

## Part 2 — Health check: Sample Ratio Mismatch (SRM)

Before reading effects, check that the bucketing itself was honest. The experiment was designed for a **50 / 50** split. If the realised split deviates from that by more than chance, the bucketing is broken and any downstream causal claim is suspect.

Run a one-sided $\chi^2$ test against the expected $50/50$ allocation.

$$\chi^2 = \sum_i \frac{(O_i - E_i)^2}{E_i}, \qquad H_0: \text{the realised split is consistent with } 50/50$$

Use `scipy.stats.chisquare(observed, expected)`. Reference: [Sample Ratio Mismatch — Wikipedia](https://en.wikipedia.org/wiki/Sample_ratio_mismatch).

**Tasks:**
- `2_chi2` — the $\chi^2$ statistic
- `2_chi2_p` — the p-value
- `2_srm_detected` — `1` if SRM detected at $\alpha = 0.05$, `0` otherwise


In [ ]:
chi2_stat = ...
chi2_p = ...
srm_detected = ...

check_answer("2_chi2", chi2_stat)
check_answer("2_chi2_p", chi2_p)
check_answer("2_srm_detected", srm_detected)


---

## Part 3 — Analysis and interpretation

The data is healthy. Build **one results table** with all six metrics. ARPU and ARPPU sit side by side with the proxies and the guardrails, and the same table feeds both the multi-metric framework and the interpretation. For each metric compute the point estimate, standard error, 95% CI, asymptotic Z-test p-value, and significance flags at the raw $\alpha = 0.05$ and the Bonferroni-corrected $\alpha' = 0.05 / 6 \approx 0.0083$.

$$\hat\Delta = \bar Y_T - \bar Y_C, \qquad \widehat{\text{SE}}(\hat\Delta) = \sqrt{\frac{s_C^{\,2}}{n_C} + \frac{s_T^{\,2}}{n_T}}, \qquad Z = \frac{\hat\Delta}{\widehat{\text{SE}}}$$

The Bonferroni family covers all six metrics: `arpu`, `arppu`, `usage_minutes`, `is_paying`, `latency_ms`, `error_rate_flag`. For `arppu`, compute on the **payer-only** subset (`is_paying == 1`) in each group.

**Ship rule:** ship if `arpu` is significant after Bonferroni **and** neither guardrail (`latency_ms`, `error_rate_flag`) is significant after Bonferroni. The proxies (`arppu`, `usage_minutes`, `is_paying`) show the *mechanism* but do not enter the ship rule.

When the table is in front of you, write **2–3 sentences** in the markdown cell below. How do you read this picture? What story do the metrics tell together? Short is better than long.

**Tasks (self-check):**
- `3_delta`, `3_se`, `3_ci_low`, `3_ci_high`, `3_p`, `3_decision` for the `arpu` row
- `4_p_arppu`, `4_p_usage`, `4_p_paying`, `4_p_error` for the other rows
- `4_alpha_bonf`, `4_n_reject_raw`, `4_n_reject_corr`, `4_ship`


In [ ]:
metrics = ["arpu", "arppu", "usage_minutes", "is_paying", "latency_ms", "error_rate_flag"]
alpha_bonf = 0.05 / len(metrics)

# Build one readout table covering all 6 metrics, with arppu computed on the
# payer-only subset of each group. Columns: delta, se, ci_low, ci_high, p,
# sig_raw, sig_bonf.

readout = ...

arpu_row = ...
delta_hat = ...
se_delta = ...
ci_low = ...
ci_high = ...
z_p_arpu = ...
decision = ...

n_reject_raw = ...
n_reject_corr = ...
ship = ...

check_answer("3_delta", delta_hat)
check_answer("3_se", se_delta)
check_answer("3_ci_low", ci_low)
check_answer("3_ci_high", ci_high)
check_answer("3_p", z_p_arpu)
check_answer("3_decision", decision)
check_answer("4_p_arppu", float(readout.loc[readout.metric == "arppu", "p"].iloc[0]))
check_answer("4_p_usage", float(readout.loc[readout.metric == "usage_minutes", "p"].iloc[0]))
check_answer("4_p_paying", float(readout.loc[readout.metric == "is_paying", "p"].iloc[0]))
check_answer("4_p_error", float(readout.loc[readout.metric == "error_rate_flag", "p"].iloc[0]))
check_answer("4_alpha_bonf", alpha_bonf)
check_answer("4_n_reject_raw", n_reject_raw)
check_answer("4_n_reject_corr", n_reject_corr)
check_answer("4_ship", ship)


### Interpretation

Write **2–3 sentences** below explaining how you read the table. What story do the metrics tell together? *(Replace this paragraph with your answer.)*


---

## Part 4 — CUPED: variance reduction table

CUPED is a variance reduction technique. If a pre-experiment covariate $X$ is correlated with the outcome $Y$, subtract the part of $Y$ that $X$ explains. The difference of means stays the same in expectation while the variance shrinks by approximately $(1 - \rho^2)$, where $\rho = \text{Corr}(X, Y)$.

$$Y_\text{cuped} = Y - \theta (X - \bar X), \qquad \theta = \frac{\widehat{\text{Cov}}(Y, X)}{\widehat{\text{Var}}(X)}$$

Build a **separate CUPED table** with two candidate outcome metrics and `usage_pre` as the pre-experiment covariate for both. Use the **full sample** (both groups) to compute $\rho$ and to fit $\theta$.

| Y | X | $\rho(X, Y)$ | $1 - \rho^2$ | $\text{Var}(Y_\text{cuped}) / \text{Var}(Y)$ |
|---|---|---|---|---|
| `usage_minutes` | `usage_pre` | ... | ... | ... |
| `revenue` | `usage_pre` | ... | ... | ... |

The grader only checks the row for `usage_minutes` (the one where CUPED is supposed to shine). The `revenue` row is for you: see what happens when the covariate has little to no relationship with the outcome.

Decision rule for `usage_minutes`: CUPED is useful if $\text{Var}(Y_\text{cuped}) / \text{Var}(Y) < 0.90$ (variance reduced by more than 10%).

**Tasks:**
- `5_rho`, `5_var_ratio_theory`, `5_var_ratio_emp` — values for `usage_minutes` with `usage_pre`
- `5_useful` — `1` if reduction $>$ 10%, else `0`


In [ ]:
def cuped_summary(X, Y):
    # Return (rho, var_ratio_theory, var_ratio_empirical) for outcome Y, covariate X.
    ...

X = df["usage_pre"].to_numpy()
candidates = [
    ("usage_minutes", df["usage_minutes"].to_numpy()),
    ("revenue",       df["revenue"].to_numpy()),
]

cuped_table = ...  # DataFrame with rows for each candidate and columns: metric, rho, var_ratio_theory, var_ratio_emp

usage_row = ...
rho = ...
var_ratio_theory = ...
var_ratio_emp = ...
useful = ...

check_answer("4_cuped_rho", rho)
check_answer("4_cuped_var_ratio_theory", var_ratio_theory)
check_answer("4_cuped_var_ratio_emp", var_ratio_emp)
check_answer("4_cuped_useful", useful)


---

## ★ Part 5 — Simulation: peeking inflates the FPR

*Parts 5 and 6 are **starred**: they validate the procedure-level intuitions from Parts 1–4 numerically. They do not change the ship-or-kill decision. Treat them as a sanity check on the framework.*

> **Data note.** Parts 5 and 6 do **not** use `ab_data.csv`. Every simulation draws fresh synthetic samples from a Normal distribution. The point is to study how the testing procedure itself behaves, so we want a clean controlled setup, not the messy real data.

Set up an A/A simulation. Both groups come from the same Normal distribution, so any "significant" result is a false positive. We check the p-value daily for up to 20 days and stop as soon as $p < 0.05$. The fraction of simulations that stop within $K$ looks is the empirical FPR after $K$ peeks.

Run $M = 2000$ simulations. In each:

1. Draw $20 \times N_{\text{per day}}$ users per group from $\mathcal{N}(0, 1)$, with $N_{\text{per day}} = 100$. Both groups use the same distribution (this is A/A).
2. For each day $k = 1, \dots, 20$, run the asymptotic Z-test on the cumulative data (first $k \cdot N_{\text{per day}}$ users in each group).
3. Stop and record "rejected" on the first $k$ where $p < 0.05$.

Then, for $K = 1, 2, 5, 10, 20$, the empirical FPR after $K$ looks is the fraction of simulations that stopped at some $k \le K$.

Use `np.random.seed(20260530)` right before the simulation loop so the grader can reproduce.

**Tasks:**
- `6_fpr_K1` — empirical FPR with 1 look
- `6_fpr_K5` — empirical FPR with 5 looks
- `6_fpr_K20` — empirical FPR with 20 looks

Plot the curve (FPR vs $K$). Comment in one line on what you see.


In [ ]:
np.random.seed(20260530)
M = 2000
N_PER_DAY = 100
DAYS = 20
ALPHA = 0.05

# Simulate M A/A runs. For each run, find the first day k where p < 0.05 (or none).
# Then build empirical FPR after K looks for K in 1..DAYS.

fpr_by_K = ...  # array of length DAYS

fpr_K1 = ...
fpr_K5 = ...
fpr_K20 = ...

# Plot FPR vs K and add a horizontal line at the nominal α.

check_answer("5_fpr_K1", fpr_K1)
check_answer("5_fpr_K5", fpr_K5)
check_answer("5_fpr_K20", fpr_K20)


---

## ★ Part 6 — Simulation: power as a function of $n$

> **Data note.** Like Part 5, this part does **not** use `ab_data.csv`. Every simulation draws fresh synthetic samples from a Normal distribution with a built-in true effect.

For a fixed true effect, plot how power grows with $n$.

Setup:
- Control $\sim \mathcal{N}(0, 1)$, treatment $\sim \mathcal{N}(0.1, 1)$ (true effect $= 0.1$, $\sigma = 1$). This is A/B with a known lift.
- For $n \in \{500, 1000, 2000, 5000, 10000\}$ per group, run $M = 2000$ asymptotic Z-tests.
- Empirical power = fraction of tests with $p < 0.05$.

Use `np.random.seed(20260530)` right before the simulation loop.

`7_n_star`: the **smallest $n$ from the grid** for which empirical power $\ge 0.80$.

**Tasks:**
- `7_power_500`, `7_power_2000`, `7_power_10000`
- `7_n_star`

Plot the power curve. Notice how power climbs with $n$ and crosses 0.80 somewhere in the grid. The same MDE formula from Part 1 gives a closed-form prediction for $n^*$ at this $\sigma$ and $\Delta$. They will not match Part 1's revenue numbers, because the sim uses a different scale ($\sigma = 1$, $\Delta = 0.1$ instead of revenue's $\sigma \approx 36$, $\Delta = 0.75$). The sim is here to show the *shape* of the curve, not to validate Part 1's numbers.


In [ ]:
np.random.seed(20260530)
M = 2000
EFFECT = 0.1
N_GRID = [500, 1000, 2000, 5000, 10000]
ALPHA = 0.05

power_values = ...  # dict {n: empirical_power}

n_star = ...  # smallest n in N_GRID with power ≥ 0.8

# Plot the power curve and add a horizontal line at target power = 0.8.

check_answer("6_power_500", power_values[500])
check_answer("6_power_2000", power_values[2000])
check_answer("6_power_10000", power_values[10000])
check_answer("6_n_star", n_star)


---

## Bonus — Delta-method CI for the relative lift on revenue

Part 3 gave you a CI for $\Delta = \mu_T - \mu_C$ in **absolute** revenue units. Stakeholders often ask the same question in **percent**: *"how big is the lift, as a fraction of the baseline?"*

The natural estimator is

$$\hat\theta = \frac{\bar Y_T}{\bar Y_C} - 1$$

You cannot just divide the absolute CI by $\bar Y_C$ — that ignores the fact that $\bar Y_C$ is also a random variable. To get a proper CI on $\hat\theta$, use the **delta method**.

### Primer — delta method

For a smooth function $g$ of a vector of estimators, $g(\hat\mu_T, \hat\mu_C)$ has approximate variance (to first-order Taylor)

$$\widehat{\text{Var}}\big(g(\hat\mu_T, \hat\mu_C)\big) \;\approx\; \left(\frac{\partial g}{\partial \mu_T}\right)^2 \widehat{\text{Var}}(\hat\mu_T) + \left(\frac{\partial g}{\partial \mu_C}\right)^2 \widehat{\text{Var}}(\hat\mu_C)$$

when the two estimators are independent (separate groups). For $g(\mu_T, \mu_C) = \mu_T/\mu_C - 1$ the partials are

$$\frac{\partial g}{\partial \mu_T} = \frac{1}{\mu_C}, \qquad \frac{\partial g}{\partial \mu_C} = -\,\frac{\mu_T}{\mu_C^{\,2}}$$

so

$$\widehat{\text{Var}}(\hat\theta) \;\approx\; \frac{s_T^{\,2}}{n_T \, \mu_C^{\,2}} \;+\; \frac{\mu_T^{\,2}\, s_C^{\,2}}{n_C \, \mu_C^{\,4}}$$

Plug in $\hat\mu_C$ for $\mu_C$ and $\hat\mu_T$ for $\mu_T$. The 95% CI is then $\hat\theta \pm z_{0.975} \cdot \sqrt{\widehat{\text{Var}}(\hat\theta)}$.

References:
- [Wikipedia — Delta method](https://en.wikipedia.org/wiki/Delta_method) (gentle entry point)
- Deng, Knoblich, Lu — *Applying the Delta Method in Metric Analytics: A Practical Guide with Novel Ideas*, KDD 2018 ([PDF](https://alexdeng.github.io/public/files/kdd2018-dm.pdf)) — covers exactly the ratio/relative-lift case you are computing here

**Tasks:**
- `bonus_lift_point` — $\hat\theta$ (a fraction, e.g., $0.06$ means a $6\%$ lift)
- `bonus_lift_se` — delta-method standard error
- `bonus_lift_ci_low`, `bonus_lift_ci_high` — 95% CI bounds for $\hat\theta$


In [ ]:
rev_c = ctrl["revenue"].to_numpy()
rev_t = treat["revenue"].to_numpy()
n_c, n_t = len(rev_c), len(rev_t)

mu_C = ...
mu_T = ...
s2_C = ...
s2_T = ...

# Point estimate of the relative lift
theta_hat = ...

# Delta-method variance and SE
var_theta = ...
se_theta = ...

z975 = st.norm.ppf(0.975)
theta_ci_low = ...
theta_ci_high = ...

print(f"θ̂ (relative lift) = {theta_hat:.4f}  ({100*theta_hat:.2f}%)")
print(f"SE(θ̂) via delta method = {se_theta:.5f}")
print(f"95% CI for θ = [{theta_ci_low:.4f}, {theta_ci_high:.4f}]  "
      f"([{100*theta_ci_low:.2f}%, {100*theta_ci_high:.2f}%])")

check_answer("bonus_lift_point", theta_hat)
check_answer("bonus_lift_se", se_theta)
check_answer("bonus_lift_ci_low", theta_ci_low)
check_answer("bonus_lift_ci_high", theta_ci_high)


---

## Done

Every `check_answer(...)` above should print **PASS**. Plots should render. Submission: this notebook, with outputs preserved, via Google Classroom.
